# Fewer false alarms: where they come from, and two cheap fixes

The cross-validation of [notebook 03](03_cross_validation_s1.ipynb) showed that every detector marks too much water. Scored on a flood it was not tuned on, the VV threshold has a bias of 2 and a FAR of 0.84 on labels drawn on other satellites.

This notebook looks at **where** the false alarms fall, then tries two fixes that need no new data:

1. **Not dark before.** A pixel already dark in the pre-event pass of the same orbit is permanent water, radar shadow or tarmac, not the flood. The EMS API publishes no permanent-water layer for these products, but the reference pass shows it.
2. **A minimum size.** Isolated dark pixels (speckle, small fields) are dropped when the connected patch is smaller than *n* pixels.

Both are scored with the leave-one-activation-out cross-validation of notebook 03: every threshold and size is chosen without the flood it is scored on. **0 PU**: it reads the pairs and references of notebook 02 from the data directory.

In [ ]:
import csv
import os
from pathlib import Path

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
RESOLUTION_M = 40
CHIP = 128
SEED = 0

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-false-alarms-s1.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. The five-band dataset of notebook 02

The dataset is rebuilt from the pairs in the data directory, locally and for free. Its train/val/test column is ignored: the folds are made by activation below.

In [ ]:
from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
assert any(p.reference is not None for p in pairs), (
    "no pair with a reference: run notebook 02 first"
)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
FOLDS = sorted({r["activation"] for r in rows})
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {len(FOLDS)} folds: {', '.join(FOLDS)}"
)

## 2. Features

The eleven features of notebook 02: VV and VH in dB, their ratio, local mean and spread of VV, the pre-event VV and VH, and the changes ΔVV, ΔVH, raw and smoothed over 5×5 pixels.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    ).astype(np.float32)


X = np.stack([features(DATASET / r["image"]) for r in rows])
LABELS = np.stack([LabelRaster.from_geotiff(DATASET / r["label"]).labels for r in rows])
Y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in rows]
km2 = (RESOLUTION_M / 1000) ** 2
print(
    f"{'activation':10} {'chips':>5} {'flooded km²':>12} {'independent km²':>16} {'same image km²':>15}"
)
for a in FOLDS:
    idx = [i for i, r in enumerate(rows) if r["activation"] == a]
    wet = [int((LABELS[i] == FLOODED).sum()) for i in idx]
    ind = sum(w for i, w in zip(idx, wet, strict=True) if not rows[i]["same_image"])
    print(
        f"{a:10} {len(idx):5d} {sum(wet) * km2:12.1f} {ind * km2:16.1f} {(sum(wet) - ind) * km2:15.1f}"
    )

## 3. Three detectors, tuned on the other activations

* **VV threshold** (notebooks 01–03): water where VV < *t* dB.
* **Not dark before**: VV < *t* dB and VV_pre ≥ *t*ₚ dB, the pixel was not already dark 6–12 days before.
* **Not dark before, minimum size**: the same, keeping only connected patches of at least *n* pixels (8-connected).

In each fold *t* and *t*ₚ are chosen together for the best CSI on the training chips, then *n*. The tuning counts observed pixels only, as the scores do.

In [ ]:
from scipy.ndimage import label as connected

from copernicus_ems.ml import Scores, evaluate

OBSERVED = LABELS != IGNORE
WET = LABELS == FLOODED
EIGHT = np.ones((3, 3), dtype=bool)


def csi(chips, predicted):
    keep, wet = OBSERVED[chips], WET[chips]
    hits = int((predicted & wet).sum())
    return hits / max(int(wet.sum()) + int((predicted & keep & ~wet).sum()), 1)


def min_size(predicted, n):
    if n <= 1:
        return predicted
    labelled, _ = connected(predicted, structure=EIGHT)
    sizes = np.bincount(labelled.ravel())
    big = sizes >= n
    big[0] = False
    return big[labelled]


def detectors(t, tp, n):
    return {
        "VV threshold": lambda f: f[..., 0] < t[0],
        "not dark before": lambda f: (f[..., 0] < tp[0]) & (f[..., 5] >= tp[1]),
        "not dark before, min size": lambda f: min_size(
            (f[..., 0] < tp[0]) & (f[..., 5] >= tp[1]), n
        ),
    }


def fold(test_activation):
    train = np.array([i for i, r in enumerate(rows) if r["activation"] != test_activation])
    vv, pre = X[train][..., 0], X[train][..., 5]
    t = max(np.arange(-22, -10, 0.5), key=lambda a: csi(train, vv < a))
    tp = max(
        [(a, b) for a in np.arange(-22, -10, 0.5) for b in np.arange(-22, -8, 1.0)],
        key=lambda c: csi(train, (vv < c[0]) & (pre >= c[1])),
    )
    base = (vv < tp[0]) & (pre >= tp[1])
    n = max(
        (1, 2, 4, 8, 16, 32), key=lambda k: csi(train, np.stack([min_size(b, k) for b in base]))
    )
    return detectors((t,), tp, n), {"t": float(t), "tp": tuple(float(v) for v in tp), "n": n}

## 4. Run the folds

In [ ]:
DETECTORS = ["VV threshold", "not dark before", "not dark before, min size"]
SUBSETS = {"independent": lambda r: not r["same_image"], "same image": lambda r: r["same_image"]}
per_fold, predictions = {}, {}
for a in FOLDS:
    found, params = fold(a)
    test = [i for i, r in enumerate(rows) if r["activation"] == a]
    for name, predict in found.items():
        for i in test:
            predictions[name, i] = predict(X[i])
        for subset, keep in SUBSETS.items():
            chosen = [i for i in test if keep(rows[i])]
            per_fold[a, name, subset] = sum(
                (evaluate(predictions[name, i], Y[i]) for i in chosen), Scores.zero()
            )
    t1, t2 = params["tp"]
    print(
        f"{a}: VV < {params['t']:.1f} | not dark before: VV < {t1:.1f} and VV_pre >= {t2:.0f} dB | min size {params['n']} px"
    )

## 5. Scores, every flood counted once

In [ ]:
pooled = {
    (name, subset): sum((per_fold[a, name, subset] for a in FOLDS), Scores.zero())
    for name in DETECTORS
    for subset in SUBSETS
}
print(f"{'':27} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'F1':>6} {'bias':>6}")
for subset in SUBSETS:
    for name in DETECTORS:
        s = pooled[name, subset]
        print(
            f"{name:27} {subset:12} {s.csi:6.3f} {s.pod:6.3f} {s.far:6.3f} {s.f1:6.3f} {s.bias:6.2f}"
        )

print(
    f"\nCSI on independent labels\n{'activation':10}"
    + "".join(f"{n:>28}" for n in DETECTORS)
    + f"{'flooded km²':>13}"
)
for a in FOLDS:
    first = per_fold[a, DETECTORS[0], "independent"]
    if first.hits + first.misses == 0:
        continue
    line = "".join(f"{per_fold[a, n, 'independent'].csi:28.3f}" for n in DETECTORS)
    print(f"{a:10}{line}{(first.hits + first.misses) * km2:13.1f}")

## 6. Where the false alarms fall

Every false alarm (predicted water where EMS says dry) of the VV threshold and of the best fix, sorted by what the two passes show:

* **dark before**: VV_pre < −15 dB, already dark before the flood (permanent water, shadow, tarmac);
* **got darker**: not dark before, and ΔVV (smoothed) < −3 dB: the surface changed, as new water does, but EMS mapped it dry (water the map missed, or that came or went between the two acquisitions);
* **other**: neither; speckle and dark surfaces that did not change much.

and, apart, the share **near** the mapped flood (within 3 pixels, 120 m): edge disagreement rather than a wrong place.

In [ ]:
from scipy.ndimage import binary_dilation

CAUSES = ["dark before", "got darker", "other"]


def causes(name, subset):
    counts = dict.fromkeys([*CAUSES, "near"], 0)
    for i, r in enumerate(rows):
        if not SUBSETS[subset](r):
            continue
        false = predictions[name, i] & OBSERVED[i] & ~WET[i]
        dark = X[i][..., 5] < -15
        darker = ~dark & (X[i][..., 9] < -3)
        counts["dark before"] += int((false & dark).sum())
        counts["got darker"] += int((false & darker).sum())
        counts["other"] += int((false & ~dark & ~darker).sum())
        counts["near"] += int((false & binary_dilation(WET[i], iterations=3)).sum())
    return counts


breakdown = {(n, s): causes(n, s) for n in (DETECTORS[0], DETECTORS[-1]) for s in SUBSETS}
print(f"{'':27} {'labels':12} {'false km²':>10}" + "".join(f"{c:>13}" for c in [*CAUSES, "near"]))
for (name, subset), c in breakdown.items():
    total = sum(c[k] for k in CAUSES)
    shares = "".join(f"{c[k] / max(total, 1):13.0%}" for k in [*CAUSES, "near"])
    print(f"{name:27} {subset:12} {total * km2:10.1f}{shares}")

## 7. Figure

Pooled CSI and bias (predicted / mapped area; 1 is right) of the three detectors, on independent and same-image labels.

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"independent": "#2a78d6", "same image": "#eb6834"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
height = 0.36
for ax, (measure, label) in zip(
    axes, [("csi", "CSI (higher is better)"), ("bias", "bias (1 is right)")], strict=True
):
    for k, subset in enumerate(SUBSETS):
        y = np.arange(len(DETECTORS)) + (k - 0.5) * (height + 0.04)
        values = [getattr(pooled[n, subset], measure) or 0 for n in DETECTORS]
        ax.barh(y, values, height=height, color=COLORS[subset], label=f"{subset} labels")
        for yy, v in zip(y, values, strict=True):
            ax.text(v * 1.01 + 0.005, yy, f"{v:.2f}", va="center", fontsize=9, color="#444")
    ax.set_xlabel(label)
    ax.grid(axis="x", color="#e5e5e5")
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
axes[1].axvline(1, color="#888", linewidth=1, linestyle="--", zorder=0)
axes[0].set_xlim(0, 0.5)
axes[1].set_xlim(0, max(2.5, max(pooled[n, s].bias for n in DETECTORS for s in SUBSETS) * 1.15))
axes[0].set_yticks(np.arange(len(DETECTORS)), DETECTORS)
axes[0].invert_yaxis()
axes[1].legend(frameon=False, loc="lower right")
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **Masking what was dark before is free and safe.** It needs only the reference pass that change detection already fetches, and it removes the false alarms a single image cannot tell from a flood: permanent water, shadow, tarmac.
* **What is left is mostly change.** The remaining false alarms on independent labels got darker between the two passes. Part of them is likely water the EMS map, drawn on another satellite at another time, does not show; a score cannot separate that from a detector's mistake.
* **Thresholds, not models.** Two thresholds and a size, tuned on six floods, carry to the seventh better than the classifiers of notebooks 02–03.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*